# ORBIT First Tool Call Session Demo

This notebook now covers both the provider-native capability probe and the first ORBIT tool-call closure path on SSH vLLM.


## Probe design

This notebook checks the following layers:
1. Canonical prompt state
2. Provider-native payload shape
3. Raw HTTP response body
4. Parsed provider message / tool_calls / finish_reason
5. ORBIT normalized interpretation
6. ORBIT closure path (tool execution + transcript reinjection + final answer)


In [26]:
from pathlib import Path
import requests

from orbit.models import ConversationMessage, MessageRole
from orbit.notebook import session_messages_dataframe
from orbit.notebook.providers.tool_call_debug import (
    build_tool_call_probe_backend,
    probe_tool_call_request,
    resolve_probe_base_url,
    run_tool_call_closure,
    tool_call_closure_summary_frame,
    tool_call_probe_summary_frame,
)

REPO_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().resolve().name == 'providers' else Path.cwd().resolve()
API_URL = 'http://127.0.0.1:8000/v1/chat/completions'
MODEL = 'Salesforce/Llama-xLAM-2-8b-fc-r'
TEST_PROMPTS = [
    'Use the read_file tool to read notes/scaffold/demo.txt. Do not answer from memory. After using the tool, tell me the first sentence only.',
    'You must use the read_file tool before answering. If you do not use the tool, your answer is incorrect. Read notes/scaffold/demo.txt and return only the first sentence.',
    'Use the read_file tool with path="notes/scaffold/demo.txt". Do not answer without calling the tool first.',
]
PROMPT = TEST_PROMPTS[0]
backend = build_tool_call_probe_backend(
    workspace_root=REPO_ROOT,
    remote_base_url='http://127.0.0.1:8000/v1',
    model=MODEL,
    api_key='EMPTY',
    auto_tunnel=True,
    ssh_host='ws3',
)
messages = [ConversationMessage(session_id='session:tool-probe', role=MessageRole.USER, content=PROMPT, turn_index=1)]
payload = backend.build_request_payload_from_messages(messages)
READY_BASE_URL = resolve_probe_base_url(backend=backend)
READY_API_URL = READY_BASE_URL.rstrip('/') + '/chat/completions'


ValidationError: 1 validation error for ConversationMessage
content
  Input should be a valid string [type=string_type, input_value=['Use the read_file tool ...alling the tool first.'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type

## 1. Canonical prompt state


In [16]:
PROMPT


'Use the read_file tool to read notes/scaffold/demo.txt. Do not answer from memory. After using the tool, tell me the first sentence only.'

## 2. Provider-assembled payload


In [17]:
payload


{'model': 'Salesforce/Llama-xLAM-2-8b-fc-r',
 'messages': [{'role': 'user',
   'content': 'Use the read_file tool to read notes/scaffold/demo.txt. Do not answer from memory. After using the tool, tell me the first sentence only.'}],
 'stream': False,
 'tools': [{'type': 'function',
   'function': {'name': 'read_file',
    'description': 'Read a file from the ORBIT workspace.',
    'parameters': {'type': 'object',
     'properties': {'path': {'type': 'string',
       'description': 'Workspace-relative file path.'}},
     'required': ['path']}}},
  {'type': 'function',
   'function': {'name': 'write_file',
    'description': 'Write text content to a file in the ORBIT workspace.',
    'parameters': {'type': 'object',
     'properties': {'path': {'type': 'string',
       'description': 'Workspace-relative file path.'},
      'content': {'type': 'string', 'description': 'File content to write.'}},
     'required': ['path', 'content']}}}],
 'tool_choice': 'auto'}

## 3. Raw provider-native HTTP response


In [18]:
response = requests.post(
    READY_API_URL,
    headers={
        'Authorization': 'Bearer EMPTY',
        'Content-Type': 'application/json',
    },
    json=payload,
    timeout=60,
)
response.status_code


200

In [19]:
raw_text = response.text
raw_text[:4000]


'{"id":"chatcmpl-a1e414fe035bc2c9","object":"chat.completion","created":1775087632,"model":"Salesforce/Llama-xLAM-2-8b-fc-r","choices":[{"index":0,"message":{"role":"assistant","content":null,"refusal":null,"annotations":null,"audio":null,"function_call":null,"tool_calls":[{"id":"call_0_b8234a30f5056ba5","type":"function","function":{"name":"read_file","arguments":"{\\"path\\": \\"notes/scaffold/demo.txt\\"}"}}],"reasoning":null},"logprobs":null,"finish_reason":"tool_calls","stop_reason":null,"token_ids":null}],"service_tier":null,"system_fingerprint":null,"usage":{"prompt_tokens":420,"total_tokens":443,"completion_tokens":23,"prompt_tokens_details":null},"prompt_logprobs":null,"prompt_token_ids":null,"kv_transfer_params":null}'

## 4. Parsed provider message / tool_calls / finish_reason


In [20]:
data = response.json()
data


{'id': 'chatcmpl-a1e414fe035bc2c9',
 'object': 'chat.completion',
 'created': 1775087632,
 'model': 'Salesforce/Llama-xLAM-2-8b-fc-r',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': None,
    'refusal': None,
    'annotations': None,
    'audio': None,
    'function_call': None,
    'tool_calls': [{'id': 'call_0_b8234a30f5056ba5',
      'type': 'function',
      'function': {'name': 'read_file',
       'arguments': '{"path": "notes/scaffold/demo.txt"}'}}],
    'reasoning': None},
   'logprobs': None,
   'finish_reason': 'tool_calls',
   'stop_reason': None,
   'token_ids': None}],
 'service_tier': None,
 'system_fingerprint': None,
 'usage': {'prompt_tokens': 420,
  'total_tokens': 443,
  'completion_tokens': 23,
  'prompt_tokens_details': None},
 'prompt_logprobs': None,
 'prompt_token_ids': None,
 'kv_transfer_params': None}

In [21]:
choice = data['choices'][0]
{
    'finish_reason': choice.get('finish_reason'),
    'message': choice.get('message'),
}


{'finish_reason': 'tool_calls',
 'message': {'role': 'assistant',
  'content': None,
  'refusal': None,
  'annotations': None,
  'audio': None,
  'function_call': None,
  'tool_calls': [{'id': 'call_0_b8234a30f5056ba5',
    'type': 'function',
    'function': {'name': 'read_file',
     'arguments': '{"path": "notes/scaffold/demo.txt"}'}}],
  'reasoning': None}}

## 5. ORBIT normalized interpretation


In [22]:
probe_result = probe_tool_call_request(backend=backend, prompt=PROMPT)
probe_result['ready_base_url'], tool_call_probe_summary_frame(probe_result)


('http://127.0.0.1:8000/v1',
              ready_base_url             plan_label source_backend  \
 0  http://127.0.0.1:8000/v1  ssh-vllm-tool-request       ssh-vllm   
 
    has_tool_request  tool_name  requires_approval side_effect_class  \
 0              True  read_file              False              safe   
 
   failure_reason  
 0           None  )

In [23]:
probe_result['plan'].model_dump()


{'source_backend': 'ssh-vllm',
 'plan_label': 'ssh-vllm-tool-request',
 'final_text': None,
 'tool_request': {'tool_name': 'read_file',
  'input_payload': {'path': 'notes/scaffold/demo.txt'},
  'requires_approval': False,
  'side_effect_class': 'safe'},
 'should_finish_after_tool': False,
 'failure_reason': None}

## 6. ORBIT closure path (tool execution + transcript reinjection + final answer)


In [24]:
closure_result = run_tool_call_closure(workspace_root=REPO_ROOT, backend=backend, prompt=PROMPT)
tool_call_closure_summary_frame(closure_result)


,session_id,plan_label,source_backend,final_text,message_count,last_message_role,last_message_kind
0,session_fc4144846bd6,ssh-vllm-final-text,ssh-vllm,"The first sentence of the file is: ""ORBIT demo...",3,assistant,None


In [25]:
session_messages_dataframe(closure_result['coordinator'].store, closure_result['session'].session_id)


,message_id,session_id,turn_index,role,content,created_at,provider_message_id,metadata
0,msg_d6fc0c12af66,session_fc4144846bd6,1,user,Use the read_file tool to read notes/scaffold/...,2026-04-01 23:54:14.100109+00:00,None,{}
1,msg_bcb1a9cfcd65,session_fc4144846bd6,2,user,Tool result from read_file:\nORBIT demo output\n,2026-04-01 23:54:15.255103+00:00,None,"{'message_kind': 'tool_result', 'tool_name': '..."
2,msg_aa8d0298a2e2,session_fc4144846bd6,3,assistant,"The first sentence of the file is: ""ORBIT demo...",2026-04-01 23:54:16.075240+00:00,None,"{'source_backend': 'ssh-vllm', 'plan_label': '..."


In [13]:
closure_result['final_plan'].model_dump()


{'source_backend': 'ssh-vllm',
 'plan_label': 'ssh-vllm-tool-request',
 'final_text': None,
 'tool_request': {'tool_name': 'read_file',
  'input_payload': {'path': 'notes/scaffold/demo.txt'},
  'requires_approval': False,
  'side_effect_class': 'safe'},
 'should_finish_after_tool': False,
 'failure_reason': None}